In [11]:
import atom_numbering as an
import pandas as pd
import get_xyz_descriptors_original as xyz
import os
import numpy as np

In [2]:
#Provide path to directory or directory of directies containing xyz files
path = "/home/aconner/orca_descriptors/AATATIAROAAKIN-SNVBAGLBSA-N/"

#Compile dataframe with atom numberings; Can specify 0 or 1 indexing for atom numberings and whether to search subdirectories or main diertory only defaults to 1 indexing and searching subdirectories
df = an.compile_final(path) 
df

,xyz_name,path,ligand_type,inchi,SCF_energy,smiles,H1,Ni,H2,N1,C1,C2,N2,ligand_class
0,optimization,/home/aconner/orca_descriptors/AATATIAROAAKIN-...,core_5,YHBAIXLGXQIEBR-SNVBAGLBSA-N,-2497.275344,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox
1,optimization,/home/aconner/orca_descriptors/AATATIAROAAKIN-...,core_5,YHBAIXLGXQIEBR-SNVBAGLBSA-N,-2497.275319,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,15,6,5,4,pyox


In [3]:
#Define lists for decriptor retrieval 
atoms = ["N1", "N2", "Ni"]
bonds = [("C1", "C2"), ("Ni", "H2")] #if order matters for 2-atom descriptors (e.g. Sterimol B1), atoms must be listed in desired order
angles = [("N1", "Ni", "N2"), ("C1", "N1", "Ni"), ("C2", "N2", "Ni")]
dihedrals = [("C1", "N1", "Ni", "N2"), ("C2", "N2", "Ni", "N1")]
quadrants = [[2, 1, 3, 4, 6]]

In [4]:
#generate descriptor dictionary for each row in dataframe and add to dataframe
results_dict = {}
for row in df.itertuples():

    P = xyz.PropertiesXYZ(path=df.loc[row.Index, "path"])
    
    #Zero atom properties
    P.get_SASA_and_Volume()

    #One atom properties
    atom_list = []
    for atom in atoms:
        atom_list.append(int(df.loc[row.Index, atom]))
    
    P.define_atom_list(atom_list)
    P.get_Vbur()
    P.get_atomic_SASA()

    #Two atom properties
    bond_list = []
    for bond in bonds: 
        bond_list.append([int(df.loc[row.Index, bond[0]]), int(df.loc[row.Index, bond[1]])])
    P.define_bond_list(bond_list)
    P.get_bond_distances()
    P.get_Sterimol_parameters()

    #Three atom properties
    angle_list = []
    for angle in angles:
        angle_list.append([int(df.loc[row.Index, angle[0]]), int(df.loc[row.Index, angle[1]]), int(df.loc[row.Index, angle[2]])])
    P.define_angle_list(angle_list)
    P.get_angles()

    #Four atom properties
    dihedral_list = []
    for dihedral in dihedrals:
        dihedral_list.append([int(df.loc[row.Index, dihedral[0]]), int(df.loc[row.Index, dihedral[1]]), int(df.loc[row.Index, dihedral[2]]), int(df.loc[row.Index, dihedral[3]])])
    P.define_dihedral_list(dihedral_list)
    P.get_dihedrals()
    #P.get_pyramidalizations()

    #Quadrant properties
    quadrant_list = quadrants
    P.define_quadrant_list(quadrant_list)
    #P.get_vbur_quadrants()
    #P.get_vbur_octants()

    results_dict[row.Index] = P.results



No xyz.json file found in the specified path. Results dictionary will be empty.
   /home/aconner/orca_descriptors/AATATIAROAAKIN-SNVBAGLBSA-N/conf_2/optimization.xyz / Bmin:  2.35 / Bmax:  5.07 / L:  7.25
   /home/aconner/orca_descriptors/AATATIAROAAKIN-SNVBAGLBSA-N/conf_2/optimization.xyz / Bmin:  2.59 / Bmax:  7.64 / L:  4.15
No xyz.json file found in the specified path. Results dictionary will be empty.
   /home/aconner/orca_descriptors/AATATIAROAAKIN-SNVBAGLBSA-N/conf_1/optimization.xyz / Bmin:  2.48 / Bmax:  5.09 / L:  7.25
   /home/aconner/orca_descriptors/AATATIAROAAKIN-SNVBAGLBSA-N/conf_1/optimization.xyz / Bmin:  2.82 / Bmax:  6.86 / L:  3.70


In [5]:
dict_atoms = {}
for row in df.itertuples():
    dict_atoms[row.Index] = {}
    for col in df.columns:
        if col.startswith("H1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("H2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("C1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("C2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("N1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("N2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("Ni"):
            atom = col  
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)



In [6]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
#Have to add new descriptors to this code!!
new_results = {}

for key in results_dict.keys():
    new_results[key] = {}
    for subkey in results_dict[key].keys():
        if subkey.startswith("Bond_"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Bond_" + bond[0] + "_" + bond[1]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_L"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_L"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmin"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmin"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmax"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmax"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Angle_"):
            for angle in angles:
                angle_key = [dict_atoms[key][angle[0]], dict_atoms[key][angle[1]], dict_atoms[key][angle[2]]]
                if all(ak in subkey for ak in angle_key):
                    new_key = "Angle_" + angle[0] + "_" + angle[1] + "_" + angle[2]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Dihed_"):
            for dihedral in dihedrals:
                dihedral_key = [dict_atoms[key][dihedral[0]], dict_atoms[key][dihedral[1]], dict_atoms[key][dihedral[2]], dict_atoms[key][dihedral[3]]]
                if all(dk in subkey for dk in dihedral_key):
                    new_key = "Dihed_" + dihedral[0] + "_" + dihedral[1] + "_" + dihedral[2] + "_" + dihedral[3]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("%Vbur_atom_"):
            for atom in atoms:
                atom_key = dict_atoms[key][atom]
                if atom_key in subkey:
                    new_key = "%Vbur_" + atom
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("SASA_"):
            for atom in atoms:
                atom_key = dict_atoms[key][atom]
                if atom_key in subkey:
                    new_key = "SASA_" + atom
                    new_results[key][new_key] = results_dict[key][subkey]
        else:
            new_results[key][subkey] = results_dict[key][subkey]

new_results

{0: {'SASA': 478.85780379711827,
  '%Vbur_N1': 44.03519283529375,
  '%Vbur_N2': 48.0162640724039,
  '%Vbur_Ni': 37.04254832707893,
  'SASA_N1': 1.0299890771726778,
  'SASA_N2': 0.45999512184410896,
  'SASA_Ni': 27.31904564345216,
  'Bond_C1_C2': 1.457737357659361,
  'Bond_Ni_H2': 1.4662339607658508,
  'Sterimol_C1_C2_Bmin': 2.353797526575055,
  'Sterimol_C1_C2_Bmax': 5.074692108887001,
  'Sterimol_C1_C2_L': 7.249999999999999,
  'Sterimol_Ni_H2_Bmin': 2.588149522297556,
  'Sterimol_Ni_H2_Bmax': 7.638880808076534,
  'Sterimol_Ni_H2_L': 4.1499999999999995,
  'Angle_N1_Ni_N2': 82.87502229970895,
  'Angle_C1_N1_Ni': 113.31895328862156,
  'Angle_C2_N2_Ni': 111.51021573295519,
  'Dihed_C1_N1_Ni_N2': -7.601367513635839,
  'Dihed_C2_N2_Ni_N1': 8.55978955179489},
 1: {'SASA': 478.6878043740325,
  '%Vbur_N1': 44.02356437829145,
  '%Vbur_N2': 48.05016398095298,
  '%Vbur_Ni': 37.034861719907916,
  'SASA_N1': 1.0399889711258126,
  'SASA_N2': 0.41999554603158146,
  'SASA_Ni': 27.279047040753113,
  'B

In [21]:
#add new results to dataframe
for key in new_results.keys():
    for subkey in new_results[key].keys():
        df.at[key, subkey] = new_results[key][subkey]

df

,xyz_name,path,ligand_type,inchi,SCF_energy,Relative_energy,smiles,H1,Ni,H2,...,Sterimol_C1_C2_Bmax,Sterimol_C1_C2_L,Sterimol_Ni_H2_Bmin,Sterimol_Ni_H2_Bmax,Sterimol_Ni_H2_L,Angle_N1_Ni_N2,Angle_C1_N1_Ni,Angle_C2_N2_Ni,Dihed_C1_N1_Ni_N2,Dihed_C2_N2_Ni_N1
0,optimization,/home/aconner/orca_descriptors/AATATIAROAAKIN-...,core_5,YHBAIXLGXQIEBR-SNVBAGLBSA-N,-2497.275344,0.0,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,...,5.074692,7.25,2.588150,7.638881,4.15,82.875022,113.318953,111.510216,-7.601368,8.559790
1,optimization,/home/aconner/orca_descriptors/AATATIAROAAKIN-...,core_5,YHBAIXLGXQIEBR-SNVBAGLBSA-N,-2497.275319,0.000025,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,1,2,3,...,5.091660,7.25,2.817871,6.861669,3.70,82.843634,113.299896,111.524681,-7.854728,8.966824


In [ ]:
#Find min, max, and average of descriptors across dataset
descriptor_stats = {}

#boltzmann weighted average
#convert energy to relative energy
if 'Relative_energy' not in df.columns:
    df.insert(5, 'Relative_energy', df['SCF_energy'] - df['SCF_energy'].min())
else:
    df['Relative_energy'] = df['SCF_energy'] - df['SCF_energy'].min()


T = 298.15 #temperature in K
R = 3.1668105e-6 #Boltzmann constant in eH/K
weights = []

for row in df.itertuples():
    energy = row.Relative_energy
    weight = np.exp(-energy/(R*T))
    weights.append(weight)

normalized_weights = weights / np.sum(weights)

for column in df.iloc[:, 15:]:
    descriptor_stats[f'{column}_min'] = df[column].min()
    descriptor_stats[f'{column}_max'] = df[column].max()
    descriptor_stats[f'{column}_mean'] = np.average(df[column], weights=normalized_weights)

descriptor_stats_df = pd.DataFrame(descriptor_stats, index=[0])
if df["smiles"].nunique() == 1:
    descriptor_stats_df.insert(0, "SMILES", df["smiles"].iloc[0])
    descriptor_stats_df.insert(1, "INCHI", df["inchi"].iloc[0])
else:
    print("Multiple SMILES strings found in dataset. Please seperate before calculating descriptor statistics.")    

descriptor_stats_df


[1.0, 0.9742038971840334]
[0.50653329 0.49346671]


,SMILES,INCHI,SASA_min,SASA_max,SASA_mean,%Vbur_N1_min,%Vbur_N1_max,%Vbur_N1_mean,%Vbur_N2_min,%Vbur_N2_max,...,Angle_C1_N1_Ni_mean,Angle_C2_N2_Ni_min,Angle_C2_N2_Ni_max,Angle_C2_N2_Ni_mean,Dihed_C1_N1_Ni_N2_min,Dihed_C1_N1_Ni_N2_max,Dihed_C1_N1_Ni_N2_mean,Dihed_C2_N2_Ni_N1_min,Dihed_C2_N2_Ni_N1_max,Dihed_C2_N2_Ni_N1_mean
0,[H]C1=C(C(F)(F)F)C([H])=C([H])C(=C2N[C@@]([H])...,YHBAIXLGXQIEBR-SNVBAGLBSA-N,478.687804,478.857804,478.773915,44.023564,44.035193,44.029455,48.016264,48.050164,...,113.309549,111.510216,111.524681,111.517354,-7.854728,-7.601368,-7.726392,8.55979,8.966824,8.760648


Below here is scrap work!

In [ ]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
new_results = {}

for key in storage.keys():
    if key.startswith("Bond_"):
        for bond in bonds:
            bond_key = [dict_atoms[bond[0]], dict_atoms[bond[1]]]
            if all(bk in key for bk in bond_key):
                new_key = key.replace(bond_key[0], f"{bond[0]}").replace(bond_key[1], f"{bond[1]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
                
    elif key.startswith("Angle_"):
        for angle in angles:
            angle_key = [dict_atoms[angle[0]], dict_atoms[angle[1]], dict_atoms[angle[2]]]
            if all(ak in key for ak in angle_key):
                new_key = key.replace(angle_key[0], f"{angle[0]}").replace(angle_key[1], f"{angle[1]}").replace(angle_key[2], f"{angle[2]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
    
    elif key.startswith("Dihed_"):
        for dihedral in dihedrals:
            dihedral_key = [dict_atoms[dihedral[0]], dict_atoms[dihedral[1]], dict_atoms[dihedral[2]], dict_atoms[dihedral[3]]]
            if all(dk in key for dk in dihedral_key):
                new_key = key.replace(dihedral_key[0], f"{dihedral[0]}").replace(dihedral_key[1], f"{dihedral[1]}").replace(dihedral_key[2], f"{dihedral[2]}").replace(dihedral_key[3], f"{dihedral[3]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
    
    elif key.startswith("%Vbur_atom_"):
        for atom in atoms:
            atom_key = dict_atoms[atom]
            if atom_key in key:
                new_key = key.replace(atom_key, f"{atom}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]

    else:
        new_storage[key] = storage[key]

            
new_storage     

In [ ]:
#Retrieve descriptors and add to dataframe
x = None
for row in df.itertuples():
    P = xyz.PropertiesXYZ(path=df.loc[row.Index, "path"])
    bond_list = []
    key_list = []
    for bond in bonds:
        #set up columns for all 2 atom descriptors
        df[f'{bond[0]}_{bond[1]}_length'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_Bmin'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_Bmax'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_L'] = None
        key_list.append(f'{bond[0]}_{bond[1]}_length')   
        bond_list.append([df.loc[row.Index, bond[0]], df.loc[row.Index, bond[1]]])
    P.define_bond_list(bond_list)
    P.get_bond_distances()
    count = 0
    for bond in bond_list:
        key = 'Bond_' + f'{bond[0]}_{bond[1]}'
        df.loc[row.Index, key_list[count]] = P.results[key]
        count += 1



In [ ]:
# 
# for bond in bond_list:


    # for key in P.results.keys():
    #     for key2 in dict_atoms.keys():
    #         if key2 in key:
    #             key = key.replace(key2, dict_atoms[key2], 1)
    #     print("new key b:" +  key)
    